# Task 06 — Model load

**Wave 1** · target file: `backend/model.py` · prerequisites: none

**🎯 Goal:** `load_models()` / `get_models()` — load the baseline + image artifacts.

Part of the *2026-06-17 fused photo-z backend*. Plan & deps: `../PLAN.md` · conventions: `../README.md`.

In [ ]:
# === Setup: run me first. Hops to the repo root so `import backend` and models/ resolve,
# and pulls the model artifact(s) from GCS if missing (needs gcloud auth - see README). ===
import os, sys
p = os.path.abspath('.')
while p != '/' and not (os.path.isdir(p+'/backend') and os.path.isdir(p+'/notebooks')):
    p = os.path.dirname(p)
os.chdir(p); sys.path.insert(0, p)
print('repo root:', p)
os.makedirs('models', exist_ok=True)
if not os.path.exists('models/fake_image_model.pkl'):
    os.system('gcloud storage cp gs://macrocosm-lewagon/models/fake_image_model.pkl models/fake_image_model.pkl')
print('models:', [f for f in os.listdir('models') if f.endswith('.pkl')])

## What to build
In `backend/model.py`:
- **`load_models()`** — `joblib.load(settings.BASELINE_PATH)` (a **dict** `{'model','features',...}` → take
  its `['model']`) and `joblib.load(settings.IMAGE_MODEL_PATH)` (a bare model). Cache both in the
  module globals `_baseline`, `_image`; return `(baseline, image)`.
- **`get_models()`** — return `(baseline, image)`, calling `load_models()` on first use.

> You can dev against a tiny stand-in baseline (below) — no 657MB download needed. In `backend/model.py`
> the functions read `settings.BASELINE_PATH` / `settings.IMAGE_MODEL_PATH` (no arguments).

## 🛠️ Develop & test here first
Fill the `# TODO` so the asserts pass. **Don't** paste the answer — write it.

In [ ]:
import joblib, numpy as np, tempfile
from sklearn.linear_model import LinearRegression
from backend.fake_model import RandomRedshiftModel
TMP = tempfile.mkdtemp()
joblib.dump({"model": LinearRegression().fit(np.random.rand(30, 16), np.random.rand(30) * 0.4),
             "features": list(range(16))}, f"{TMP}/baseline.pkl")
joblib.dump(RandomRedshiftModel(0), f"{TMP}/image.pkl")

_baseline = None; _image = None
def load_models(baseline_path=f"{TMP}/baseline.pkl", image_path=f"{TMP}/image.pkl"):
    # TODO: load both; baseline pkl is a dict -> take ['model']; cache in _baseline/_image; return (baseline, image)
    raise NotImplementedError
def get_models():
    # TODO: load on first call, then return (baseline, image)
    raise NotImplementedError

b, i = load_models()
assert hasattr(b, "predict") and hasattr(i, "predict")
assert len(b.predict(np.random.rand(2, 16))) == 2
print("OK", type(b).__name__, "+", type(i).__name__)

## ➡️ Move it into `backend/model.py`
Once the cell above passes, put your implementation into **`backend/model.py`** (replace the `# TODO`). Then run the **Check** cell — it imports from `backend/` and verifies the real module.

## ✅ Check (imports from `backend/`)

In [ ]:
import os, joblib, numpy as np, tempfile, importlib
from sklearn.linear_model import LinearRegression
from backend.fake_model import RandomRedshiftModel
TMP = tempfile.mkdtemp()
joblib.dump({"model": LinearRegression().fit(np.random.rand(30, 16), np.random.rand(30)), "features": list(range(16))}, f"{TMP}/b.pkl")
joblib.dump(RandomRedshiftModel(0), f"{TMP}/i.pkl")
os.environ["BASELINE_PATH"] = f"{TMP}/b.pkl"; os.environ["IMAGE_MODEL_PATH"] = f"{TMP}/i.pkl"
import backend.config, backend.model
importlib.reload(backend.config); importlib.reload(backend.model)
b, i = backend.model.load_models()
assert hasattr(b, "predict") and hasattr(i, "predict")
print("model-load OK:", type(b).__name__)

> ⚠️ **06 + 09 both edit `backend/model.py`.** Keep *both* functions when merging — see `MERGE.md`.

## 🚀 Ship it

In [ ]:
# === Commit & push on YOUR OWN branch (run last; repo root after Setup) ===
!git checkout -B task/06-model-load
!git add backend/model.py notebooks/tasks-2026-6-17/06-model-load/task.ipynb
!git commit -m "06-model-load: Model load"
!git push -u origin task/06-model-load
# then merge back into 2026.6.17 -> see MERGE.md in this folder